# Notebook 24
## Clustering
### Unsupervised Learning
Model is given a dataset of features but no labels.

It's job is to find structures, patterns, or groupings within the data based on similarities.

### K-Clustering
Attempts to divide a dataset into k distinct, non-overlapping clusters.

K stands for the number of clusters you want to find.

Means refers to how the algo finds the center of every cluster.
- Calculating the average of all points in that cluster.

### Core Algorithm
Finds the clusters through guessing and adjusting.

- Initialization
    - User chooses the number of clusters.
    - Algorithm randomly drops k onto the data space.

- Assignment
    - Calculates the distance between every single data point and every centroid.
    - Each data point is assigned to the cluster of its closest cluster.

- Update
    - Now that the points are grouped, the algorithm recalculates the true center of each group.
    - Moves the mathematical average coordinates of all the points assigned to it.

- Repeat
    - Steps 2 and 3.
    - Algorithm stops when the centroids stop moving because the assignments no longer change.

### Weaknesses of K-Means
Rigid Boundaries
- K-Means draws straight, rigid lines between clusters.
- If data naturally overlaps or bleeds together, K-Means will still chop it in half.

Spherical Assumption
- Uses distance from a center point, K-Means assumes all clusters are spherical.
- Struggles heavily with elongated or oddly shaped data distributions.

## Choosing Number of Clusters
Must declare k upfront.

To find the best k, we plot the sume of squared distances for different values of k and look for a bend in the graph.

## Overlapping vs. Distinct Data
Distinct
- When data points from tight, isolated islands, K-Means easily draws perfect boundaries between them.

Overlapping
- When natural groups spread out and bleed into each other, K-Means struggles.
- It is forced to draw rigid, straight lines through the dense, overlapping areas.

## Experimenting with k
K-Means doesn't know the real answer so it's crucial to change the k value in unsupervised learning.

- Clustering finds a grouping based strictly on your k value, not necessarily the true grouping.

## Generating Synthetic Data
When learning algo's its highly useful to test them on perfectly controlled data before using real-world data.

- make_blobs
    - Function generates synthetic 2D or 3D datasets.
    - You can control how many samples to make, how many true centers exist, and how spread out they are.

## Image Grouping
Clustering isn't restricted to 2D, it can also group high-dimensional data like images.

- When applied to digits dataset, an 8x8 pixel image is flattened into a 64-dimensional vector.

- K-Means calculates the Euclidean distances between these 64-dimensional arrays.
    - If two images have dark pixels in the exact same spots, the distance between them is small, and K-Means will pull them into the same cluster.

## Cluster ID vs. Label Disconnect
When K-Means groups the digits dataset into 10 clusters, it assigns them

- Cluster 0 doesn't mean the number 0.
- Must manually interpret the groups, they don't mean anything just gathers stuff that's similar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, load_digits

# ==========================================
# TOPIC 1: GENERATING SYNTHETIC DATA
# ==========================================
print("--- 1. Generating Synthetic Data ---")
# We use make_blobs to create 300 fake 2D data points that naturally form 3 distinct groups.
# cluster_std=1.0 keeps the groups tight and distinct.
X_distinct, true_labels = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)
print(f"Successfully generated {X_distinct.shape[0]} synthetic data points.\n")


# ==========================================
# TOPIC 2: CLUSTERING & EXPERIMENTING WITH K
# TOPIC 3: CHOOSING THE NUMBER OF CLUSTERS
# ==========================================
print("--- 2 & 3. Clustering, Choosing 'k', & Experimenting ---")
print("We know the synthetic data has 3 true centers. Let's see what happens if we guess 'k' wrong.")

# We will test k=2, k=3 (correct), and k=5
for k in [2, 3, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    predicted_labels = kmeans.fit_predict(X_distinct)
    centers = kmeans.cluster_centers_
    
    plt.figure(figsize=(5, 3))
    plt.scatter(X_distinct[:, 0], X_distinct[:, 1], c=predicted_labels, cmap='viridis', alpha=0.6)
    plt.scatter(centers[:, 0], centers[:, 1], marker="X", s=200, c='red')
    plt.title(f"Experimenting with K: KMeans forced into {k} clusters")
    plt.show() # NOTE: Close the plot window to continue the script!

print("As seen in the plots, KMeans doesn't know the 'truth'. It just blindly follows the 'k' you give it.\n")


# ==========================================
# TOPIC 4: OVERLAPPING VS. DISTINCT DATA
# ==========================================
print("--- 4. Overlapping vs. Distinct Data ---")
# We generate new data, but increase cluster_std to 2.5 so the 3 groups bleed into each other.
X_overlap, _ = make_blobs(n_samples=300, centers=3, cluster_std=2.5, random_state=42)

kmeans_overlap = KMeans(n_clusters=3, random_state=42, n_init=10)
overlap_labels = kmeans_overlap.fit_predict(X_overlap)
overlap_centers = kmeans_overlap.cluster_centers_

plt.figure(figsize=(5, 3))
plt.scatter(X_overlap[:, 0], X_overlap[:, 1], c=overlap_labels, cmap='viridis', alpha=0.6)
plt.scatter(overlap_centers[:, 0], overlap_centers[:, 1], marker="X", s=200, c='red')
plt.title("Overlapping Data: Notice the rigid mathematical boundaries")
plt.show()

print("Notice how KMeans draws straight, rigid lines right through the dense overlapping areas.\n")


# ==========================================
# TOPIC 5: IMAGE GROUPING
# ==========================================
print("--- 5. Image Grouping (High-Dimensional Data) ---")
# We load the digits dataset. Each image is 64 pixels (64 dimensions).
digits = load_digits()
X_digits = digits.data  
images = digits.images  
y_true_digits = digits.target  

print(f"Loaded {X_digits.shape[0]} images, flattened into {X_digits.shape[1]} dimensions.")

# We cluster the images into 10 groups, hoping it finds the 10 numbers.
kmeans_digits = KMeans(n_clusters=10, random_state=42, n_init=10)
digit_clusters = kmeans_digits.fit_predict(X_digits)
print("Clustering of 64-dimensional images complete!\n")


# ==========================================
# TOPIC 6: CLUSTER ID VS. LABEL DISCONNECT
# ==========================================
print("--- 6. Cluster ID vs. Label Disconnect (The Big Trap!) ---")
print("Let's look at the first 15 images KMeans put into its internal 'Cluster 0'.")

# Find the indices of all images assigned to Cluster 0
cluster_0_indices = np.where(digit_clusters == 0)[0]

# Print the true, human-readable labels of the images that KMeans dumped into "Cluster 0"
print(f"True labels of the first 15 images in 'Cluster 0':")
print(y_true_digits[cluster_0_indices[:15]])

print("\nDANGER: KMeans named it 'Cluster 0', but as you can see, the actual images inside")
print("are mostly the digit '4' (with maybe a few mistakes).")
print("The algorithm doesn't know what a 4 or a 0 is. It just grouped visually similar pixels together!")